## Modelo predictivo de coste de seguro

## Cargamos el CSV y las librerias

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_validate, learning_curve
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from scipy.stats import loguniform
from sklearn.linear_model import Ridge
from sklearn.linear_model import RidgeCV

In [2]:
df= pd.read_csv("Data/df_con_probabilidad.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101886 entries, 0 to 101885
Data columns (total 24 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   ID                                     101886 non-null  str    
 1   Edad                                   101886 non-null  int64  
 2   Ingresos                               101886 non-null  float64
 3   Monto_Inicial                          101886 non-null  int64  
 4   Scoring_Crediticio                     101886 non-null  int64  
 5   Meses_Empleo                           101886 non-null  float64
 6   Num_Creditos                           101886 non-null  int64  
 7   Ratio_Interes                          101886 non-null  float64
 8   Duracion                               101886 non-null  float64
 9   Ratio_Deuda_Ingresos                   101886 non-null  float64
 10  Estudios                               101886 non-null  int64  
 11

In [4]:
df= df.drop(columns="Impago")

Eliminamos la columna impago porque es una clasificacion poco realista, Esto es debido a que la clasificacion depende de un umbral que nosotros mismos hemos elegido a proposito para que trate de detectar el maximo numero de impagos posibles. Es por eso que vamos a trabajar unicamente con la probabilidad de impago que nuestro modelo nos ha devuelto

Aun asi viendo las variables me he dado cuenta que a la hora de hacer un get dummies de las variables categoricas se nos olvido eliminar una columna por lo que vamos a eliminar una de cada una. Esto es necesario porque de no hacerlo la suma de las categorias siempre sera 1 porque imaginate si tengo tres categorias (en formato dummy) y se dos de ellas no necesito la tercera 

Para ello he decidido eliminar la mas frecuente en ambas para que sea una referencia intuitiva

In [5]:
cols_estado_civil = ["Estado_Civil_Casado", "Estado_Civil_Divorciado", "Estado_Civil_Soltero"]
cols_jornada = ["Tipo_Jornada_Laboral_Autónomo", "Tipo_Jornada_Laboral_Desempleado",
                "Tipo_Jornada_Laboral_Jornada completa", "Tipo_Jornada_Laboral_Tiempo parcial"]

In [6]:
estado_civil_media = df[cols_estado_civil].mean()
print(estado_civil_media)

Estado_Civil_Casado        0.511778
Estado_Civil_Divorciado    0.186424
Estado_Civil_Soltero       0.301798
dtype: float64


In [7]:
df= df.drop(columns="Estado_Civil_Casado")

In [8]:
jornada_media = df[cols_jornada].mean()
print(jornada_media)

Tipo_Jornada_Laboral_Autónomo            0.148519
Tipo_Jornada_Laboral_Desempleado         0.099513
Tipo_Jornada_Laboral_Jornada completa    0.550056
Tipo_Jornada_Laboral_Tiempo parcial      0.201912
dtype: float64


In [9]:
df= df.drop(columns="Tipo_Jornada_Laboral_Jornada completa")

Vamos a eliminar la columna ID porque es un identificador y nos nos sirve en el modelo

In [10]:
df = df.drop(columns="ID")

Vamos a ver las correlaciones de las variables

In [11]:
corr = df.corr().abs().stack()
corr[(corr > 0.7) & (corr < 1)]

Edad          Meses_Empleo    0.941027
              Prob_Impago     0.728869
Meses_Empleo  Edad            0.941027
Prob_Impago   Edad            0.728869
dtype: float64

Hemos observado que Edad presenta una alta correlación tanto con Meses_Empleo (0.941) como con Prob_Impago (0.729). Sin embargo, eliminar Prob_Impago sería inviable debido a su importancia en el cálculo de la prima. Por ello, probaremos a eliminar Edad, manteniendo las otras dos variables.

In [12]:
df = df.drop(columns="Edad")

## Separamos en TRAIN TEST y PUBLICO

In [13]:
df_modelo = df.copy()



df_publico = df_modelo.sample(frac=0.10, random_state=42)
df_modelo = df_modelo.drop(df_publico.index)


target = "Prima"
X = df_modelo.drop(columns=target)
y = df_modelo[target]

X_publico = df_publico.drop(columns=target)
y_publico = df_publico[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Regresion simple

In [14]:
columnas_a_escalar = ["Meses_Empleo", "Ingresos", "Monto_Inicial", "Scoring_Crediticio",
                      "Num_Creditos", "Ratio_Interes", "Duracion", "Ratio_Deuda_Ingresos"]

scaler = StandardScaler()

X_train_scal = X_train.copy()
X_test_scal = X_test.copy()
X_publico_scal = X_publico.copy()

X_train_scal[columnas_a_escalar] = scaler.fit_transform(X_train[columnas_a_escalar])
X_test_scal[columnas_a_escalar] = scaler.transform(X_test[columnas_a_escalar])
X_publico_scal[columnas_a_escalar] = scaler.transform(X_publico[columnas_a_escalar])

In [15]:
X_train_scaled = sm.add_constant(X_train_scal, prepend=True)
modelo_lin_reg = sm.OLS(endog=y_train, exog=X_train_scaled)
modelo_lin_reg = modelo_lin_reg.fit()
print(modelo_lin_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  Prima   R-squared:                       0.677
Model:                            OLS   Adj. R-squared:                  0.677
Method:                 Least Squares   F-statistic:                     8547.
Date:                Wed, 04 Nov 2026   Prob (F-statistic):               0.00
Time:                        15:38:20   Log-Likelihood:            -4.7249e+05
No. Observations:               73357   AIC:                         9.450e+05
Df Residuals:                   73338   BIC:                         9.452e+05
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
co

Vamos a quitar las variables con un p valor mas alto que 0,05 ya que si este es mas alto que 0,05 no podriamos rechazar la hipotesis nula y por lo tanto esas variables no serian estadisticamente significativas

In [16]:
variables_quitar = [
    "Meses_Empleo",
    "Num_Creditos",
    "Posesion_Hipoteca",
    "Personas_Cargo",
    "Fiador",
    "Estado_Civil_Divorciado",
    "Estado_Civil_Soltero",
    "Tipo_Jornada_Laboral_Autónomo",
    "Tipo_Jornada_Laboral_Desempleado",
    "Tipo_Jornada_Laboral_Tiempo parcial"
    
]

In [17]:
X_train_hipotesisnula =X_train_scal.drop(columns=variables_quitar)
X_test_hipotesisnula = X_test_scal.drop(columns=variables_quitar)
X_publico_hipotesisnula =X_publico_scal.drop(columns=variables_quitar)

In [18]:
X_train_scaled = sm.add_constant(X_train_hipotesisnula, prepend=True)
modelo_lin_reg = sm.OLS(endog=y_train, exog=X_train_scaled)
modelo_lin_reg = modelo_lin_reg.fit()
print(modelo_lin_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  Prima   R-squared:                       0.677
Model:                            OLS   Adj. R-squared:                  0.677
Method:                 Least Squares   F-statistic:                 1.922e+04
Date:                Wed, 04 Nov 2026   Prob (F-statistic):               0.00
Time:                        15:38:20   Log-Likelihood:            -4.7252e+05
No. Observations:               73357   AIC:                         9.450e+05
Df Residuals:                   73348   BIC:                         9.451e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                  402.7076 

## Regresion Polinomica

En la regresión polinomica los datos añadiendo grados al polinomio el modelo se ajusta a la curva de los datos por lo que vamos a ver cual es el mejor grado para nuestro modelo

In [19]:
def comparar_grados(X, y, grados=(3,4, 5)):

    
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

   
    resultados = []

    
    for g in grados:
        
        poly = PolynomialFeatures(degree=g, include_bias=False)
        X_poly = poly.fit_transform(X)

       
        scores = cross_validate(LinearRegression(), X_poly, y, cv=cv, return_train_score=True,
                                scoring={"rmse": "neg_root_mean_squared_error",
                                         "mae": "neg_mean_absolute_error",
                                         "r2": "r2"})

   
        resultados.append({
            "grado": g,
            "n_variables": X_poly.shape[1],               
            "RMSE_train": -scores["train_rmse"].mean(),   
            "RMSE_cv": -scores["test_rmse"].mean(),       
            "MAE_cv": -scores["test_mae"].mean(),         
            "R2_cv": scores["test_r2"].mean()             
        })

   
    return pd.DataFrame(resultados)

resultados = comparar_grados(X_train_hipotesisnula, y_train)
resultados

,grado,n_variables,RMSE_train,RMSE_cv,MAE_cv,R2_cv
0,3,164,74.862416,75.174614,55.986323,0.920746
1,4,494,63.560379,64.375070,47.077249,0.941882
2,5,1286,57.352292,59.478422,42.434641,0.950386


Hemos elegido como mejor modelo el de grado 5 ahora vamos a ver sus metricas de error 

In [20]:
mejor_grado = int(resultados.loc[resultados["RMSE_cv"].idxmin(), "grado"])

# Generamos las variables polinómicas con el grado elegido
poly = PolynomialFeatures(degree=mejor_grado, include_bias=False)
X_train_poly = poly.fit_transform(X_train_hipotesisnula)  
X_test_poly = poly.transform(X_test_hipotesisnula)        

# Entrenamos el modelo final con todo el conjunto de entrenamiento
modelo_final = LinearRegression()
modelo_final.fit(X_train_poly, y_train)

# Predecimos sobre el test: datos que el modelo no ha visto en ningún momento
pred_test = modelo_final.predict(X_test_poly)

# Evaluamos el modelo una única vez en test: esta es la estimación final de su rendimiento con datos nuevos
print(f"Mejor grado: {mejor_grado}")
print(f"Test:  RMSE = {root_mean_squared_error(y_test, pred_test):.2f} | "   
      f"MAE = {mean_absolute_error(y_test, pred_test):.2f} | "               
      f"R² = {r2_score(y_test, pred_test):.3f}")    

Mejor grado: 5
Test:  RMSE = 58.22 | MAE = 41.79 | R² = 0.953


Una vez elegido como mejor grado el 5 vamos añadir esas variables a X_train , X_test y X_publico 

In [21]:
poly = PolynomialFeatures(degree=5, include_bias=False)

X_train_poly = poly.fit_transform(X_train_hipotesisnula)
X_test_poly = poly.transform(X_test_hipotesisnula)
X_publico_poly = poly.transform(X_publico_hipotesisnula)




scaler = StandardScaler()

X_train_poly_scaled = scaler.fit_transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)
X_publico_poly_scaled= scaler.transform(X_publico_poly)

## Regularización 

Arriba hemos visto que sin regularizar Test:  RMSE = 58.22 | MAE = 41.79 | R² = 0.953

Definimos el cross validation cabe recalcar que se empezo probando con 5 folds pero los tiempos de ejecucion eran tan altos que se opto por reducir tanto el numero de folds como el espacio del alpha en cada regularizacion

In [22]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)


RIDGE

In [23]:


modelo_ridgecv = RidgeCV(
    alphas=np.logspace(-3, 2, 10),
    scoring="neg_root_mean_squared_error",
    cv=cv
)

modelo_ridgecv.fit(X_train_poly_scaled, y_train)

,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.For an example on how regularization strength affects the model coefficients,see :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`.",array([1.0000...00000000e+02])
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error <mean_squared_error>` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`coefficient of determination <r2_score>` (:math:`R^2`) otherwise.",'neg_root_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used, else,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.",KFold(n_split... shuffle=True)
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"gcv_mode gcv_mode: {'auto', 'svd', 'eigen'}, default='auto'Flag indicating which strategy to use when performingLeave-One-Out Cross-Validation. Options are:: 'auto' : same as 'eigen' 'svd' : use singular value decomposition of X when X is dense, fallback to 'eigen' when X is sparse 'eigen' : use eigendecomposition of X X' when n_samples <= n_features or X' X when n_features < n_samplesThe 'auto' mode is the default and is intended to pick the cheaperoption depending on the shape and sparsity of the training data.",None
,"store_cv_results store_cv_results: bool, default=FalseFlag indicating if the cross-validation values corresponding toeach alpha should be stored in the ``cv_results_`` attribute (seebelow). This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionchanged:: 1.5 Parameter name changed from `store_cv_values` to `store_cv_results`.",False
,"alpha_per_target alpha_per_target: bool, default=FalseFlag indicating whether to optimize the alpha value (picked from the`alphas` parameter list) for each target separately (for multi-outputsettings: multiple prediction targets). When set to `True`, afterfitting, the `alpha_` attribute will contain a value for each target.When set to `False`, a single alpha is used for all targets.This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionadded:: 0.24",False
Name,Type,Value
"alpha_ alpha_: float or ndarray of shape (n_targets,)Estimated regularization parameter, or, if ``alpha_per_target=True``,the estimated regularization parameter for each target.",float64,0.1668
"best_score_ best_score_: float or ndarray of shape (n_targets,)Score of base estimator with best alpha, or, if``alpha_per_target=True``, a score for each target... versionadded:: 0.23",float64,-59.84
"coef_ coef_: ndarray of shape (n_features) or (n_targets, n_features)Weight vector(s).","ndarray[flo

In [24]:
modelo_ridgecv.alpha_

np.float64(0.1668100537200059)

In [25]:
ridgeR = Ridge(alpha=modelo_ridgecv.alpha_)

ridgeR.fit(X_train_poly_scaled, y_train)

print("Train RMSE:", root_mean_squared_error(y_train, ridgeR.predict(X_train_poly_scaled)))
print("Test RMSE:", root_mean_squared_error(y_test, ridgeR.predict(X_test_poly_scaled)))

Train RMSE: 57.56084344921031
Test RMSE: 58.3044873489061


LASSO

In [26]:
from sklearn.linear_model import Lasso
from sklearn.linear_model import LassoCV
import numpy as np


modeloLasso = LassoCV(
    alphas=np.logspace(0,3,5),
    cv=cv,
    max_iter=3000,
    tol=0.005,
)

modeloLasso.fit(X_train_poly_scaled, y_train)
modeloLasso.fit(X_train_poly_scaled, y_train)

,"alphas alphas: array-like or int, default=100Values of alphas to test along the regularization path.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`.",array([ 1. ...000. ])
,"max_iter max_iter: int, default=1000The maximum number of iterations.",3000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``.",0.005
,"cv cv: int, cross-validation generator or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- int, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For int/None inputs, :class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: 'auto', bool or array-like of shape (n_features, n_features), default='auto'Whether to use a precomputed Gram matrix to speed upcalculations. If set to ``'auto'`` let us decide. The Grammatrix can also be passed as an argument.",'auto'
,"copy_X copy_X: bool, default=TrueIf ``True``, `X` will be copied; otherwise, it may be overwritten.",True
,"verbose verbose: bool or int, default=FalseAmount of verbosity.",False
,"n_jobs n_jobs: int, default=NoneNumber of CPUs to use during the cross validation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"positive positive: bool, default=FalseIf positive, restrict regression coefficients to be positive.",False


In [27]:
modeloLasso = Lasso(alpha = modeloLasso.alpha_ , max_iter=3000 , tol=0.005)
modeloLasso.fit(X_train_poly_scaled, y_train)

,"max_iter max_iter: int, default=1000The maximum number of iterations.",3000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.005
,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",np.float64(1.0)
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'
Name,Type,Value


In [28]:
print("Train RMSE:", root_mean_squared_error(y_train, modeloLasso.predict(X_train_poly_scaled)))

print("Test RMSE:", root_mean_squared_error(y_test, modeloLasso.predict(X_test_poly_scaled)))

Train RMSE: 75.80005417776968
Test RMSE: 75.78146321281228


ELASTICNET

In [29]:
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import ElasticNetCV

elastic_net = ElasticNetCV(
    alphas=np.logspace(0, 3, 5),
    l1_ratio=[0.2, 0.5, 0.8],
    cv=cv,
    max_iter=3000,
    tol=0.005
)

elastic_net.fit(X_train_poly_scaled, y_train)

,"l1_ratio l1_ratio: float or list of float, default=0.5Float between 0 and 1 passed to ElasticNet (scaling betweenl1 and l2 penalties). For ``l1_ratio = 0``the penalty is an L2 penalty. For ``l1_ratio = 1`` it is an L1 penalty.For ``0 < l1_ratio < 1``, the penalty is a combination of L1 and L2This parameter can be a list, in which case the differentvalues are tested by cross-validation and the one giving the bestprediction score is used. Note that a good choice of list ofvalues for l1_ratio is often to put more values close to 1(i.e. Lasso) and less close to 0 (i.e. Ridge), as in ``[.1, .5, .7,.9, .95, .99, 1]``.","[0.2, 0.5, ...]"
,"alphas alphas: array-like or int, default=100Values of alphas to test along the regularization path, used for each l1_ratio.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`.",array([ 1. ...000. ])
,"max_iter max_iter: int, default=1000The maximum number of iterations.",3000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``.",0.005
,"cv cv: int, cross-validation generator or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- int, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For int/None inputs, :class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: 'auto', bool or array-like of shape (n_features, n_features), default='auto'Whether to use a precomputed Gram matrix to speed upcalculations. If set to ``'auto'`` let us decide. The Grammatrix can also be passed as argument.",'auto'
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"verbose verbose: bool or int, default=0Amount of verbosity.",0
,"n_jobs n_jobs: int, default=NoneNumber of CPUs to use during the cross validation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None


In [30]:
print(f"alpha elegido: {elastic_net.alpha_}")        
print(f"l1_ratio elegido: {elastic_net.l1_ratio_}")  

alpha elegido: 1.0
l1_ratio elegido: 0.8


In [31]:
print("Train RMSE:", root_mean_squared_error(y_train, elastic_net.predict(X_train_poly_scaled)))

print("Test RMSE:", root_mean_squared_error(y_test, elastic_net.predict(X_test_poly_scaled)))

Train RMSE: 106.65431127408287
Test RMSE: 106.7552893417651


La regresión polinómica de grado 5 sin regularizar ha obtenido el mejor resultado, con un RMSE de 58.46 en test. Aunque Ridge presenta un rendimiento muy similar (58.58), no consigue mejorar el modelo original. Por otro lado, Lasso y ElasticNet presentan errores superiores, por lo que descartamos su utilización.

## Modelo Regresión Polinomica

In [32]:
X_train_poly_scaled = sm.add_constant(X_train_poly_scaled, prepend=True)
modelo_poly_reg = sm.OLS(endog=y_train, exog=X_train_poly_scaled)
modelo_poly_reg = modelo_poly_reg.fit()
print(modelo_poly_reg.summary())

C:\Users\guill\AppData\Local\Temp\ipykernel_4404\1888200478.py:3: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  modelo_poly_reg = modelo_poly_reg.fit()


                            OLS Regression Results                            
Dep. Variable:                  Prima   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.953
Method:                 Least Squares   F-statistic:                     1165.
Date:                Wed, 04 Nov 2026   Prob (F-statistic):               0.00
Time:                        15:57:04   Log-Likelihood:            -4.0123e+05
No. Observations:               73357   AIC:                         8.050e+05
Df Residuals:                   72080   BIC:                         8.168e+05
Df Model:                        1276                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        384.7176      0.214   1798.308      0.0

In [33]:
pred_train = modelo_poly_reg.predict(X_train_poly_scaled)

pred_train

array([490.80719422, 241.88193021, 578.58215178, ..., 453.30393037,
       708.28633986,  38.98414534], shape=(73357,))

Vamos a ver las metricas de train y comparar con test

Primero hay que añadir una constante al test porque al modelo se la hemos añadido y entonces no podria predecir con una variable menos

In [34]:
X_test_poly_scaled = sm.add_constant(X_test_poly_scaled)

pred_test = modelo_poly_reg.predict(X_test_poly_scaled)

In [35]:
print(f"Train: RMSE = {root_mean_squared_error(y_train, pred_train):.2f} | MAE = {mean_absolute_error(y_train, pred_train):.2f} | R² = {r2_score(y_train, pred_train):.3f}")
print(f"Test:  RMSE = {root_mean_squared_error(y_test, pred_test):.2f} | MAE = {mean_absolute_error(y_test, pred_test):.2f} | R² = {r2_score(y_test, pred_test):.3f}")

Train: RMSE = 57.44 | MAE = 41.40 | R² = 0.954
Test:  RMSE = 58.24 | MAE = 41.82 | R² = 0.953


## Comparacion del modelo final con el conjunto publico

Para comparar como se va a comportar el modelo que hemos creado con datos publicos hemos separado al principio un conjunto de datos llamado df_publico los cuales no se han tocado en todo el tiempo que hemos entrenado. Por ello vamos a comprobar el rendimiento del modelo con datos que no han sido utilizados para entrenar (como es train) o posibles filtrados de informacion que haya podido tener el conjunto test: Para comprobar el rendimiento de este modelo comparamos con las metricas de train y test.

### Train: RMSE = 57.44 | MAE = 41.40 | R² = 0.954
### Test:  RMSE = 58.24 | MAE = 41.82 | R² = 0.953

AL conjunto publico tambien añadimos una constante para tener el mismo numero de variables que en el modelo

In [36]:
X_publico_poly_scaled = sm.add_constant(X_publico_poly_scaled)

pred_final = modelo_poly_reg.predict(X_publico_poly_scaled)

In [37]:
print(f"Publico:  RMSE = {root_mean_squared_error(y_publico, pred_final):.2f} | MAE = {mean_absolute_error(y_publico, pred_final):.2f} | R² = {r2_score(y_publico, pred_final):.3f}")

Publico:  RMSE = 58.73 | MAE = 42.22 | R² = 0.951


### Publico:  RMSE = 58.73 | MAE = 42.22 | R² = 0.951

Los resultados son muy similares en los tres conjuntos. Aunque las métricas empeoran ligeramente al utilizar datos públicos, la diferencia es mínima. Esto indica que el modelo mantiene un rendimiento estable ante datos que no ha visto anteriormente y que presenta una buena capacidad de generalización, sin evidencias claras de sobreajuste.